# Analiza Forex - Midas Project

Advanced ML Analysis with XGBoost, LSTM, and Nearest Neighbors.

In [1]:
import plotly.graph_objects as go
import pandas as pd
import config
from data_connector import MT5Connector
from database import DatabaseManager
from analysis import Analyzer

# Settings
pd.set_option('display.max_columns', None)

## 1. Data Connection

In [2]:
connector = MT5Connector()
db = DatabaseManager()
symbol = 'XAUUSD'

df = db.load_candles(symbol, limit=10000)
if df.empty:
    print("No data. Run main.py first.")
else:
    print(f"Loaded {len(df)} candles for {symbol}.")

## 2. Analysis (ZigZag & Swings)

In [3]:
if not df.empty:
    df['zigzag'] = Analyzer.calculate_zigzag(df)
    swings = Analyzer.analyze_swings(df)
    print(f"Swings found: {len(swings)}")

## 3. Advanced ML Predictions

In [14]:
# Model Selection for Live Prediction
MODEL_TYPE = 'XGBoost' # Options: 'NN', 'XGBoost', 'LSTM'

prediction = None
if not swings.empty:
    print(f"Making live prediction using {MODEL_TYPE}...")
    if MODEL_TYPE == 'NN':
        prediction = Analyzer.predict_next_swing_nn(swings, k=5)
    elif MODEL_TYPE == 'XGBoost':
        prediction = Analyzer.predict_next_swing_xgboost(swings, window=10)
    elif MODEL_TYPE == 'LSTM':
        prediction = Analyzer.predict_next_swing_lstm(swings, window=10)

    if prediction:
        print(f"\n=== {MODEL_TYPE} PROGNOZA ===")
        print(f"Kierunek: {prediction['direction']}")
        print(f"Cel cenowy: {prediction['target_price']:.2f}")
        print(f"Przewidywany czas: {prediction['target_time']}")
    else:
        print("No prediction (insufficient data for this model).")

# Wave Logic Signals (SOT & Hinge)
sot_signals = Analyzer.detect_sot(swings)
hinge_signals = Analyzer.detect_hinge(swings)
if not hinge_signals.empty:
    springboard_signals = Analyzer.detect_springboard(swings, hinge_signals)
else:
    springboard_signals = pd.DataFrame()

## 4. Model Backtesting & Verification

In [ ]:
# Choose model to backtest
BT_MODEL = 'XGBoost' # Try 'NN', 'XGBoost', or 'LSTM'

backtest_results = pd.DataFrame()
if not swings.empty:
    print(f"Backtesting {BT_MODEL} (this may take a few seconds)...")
    backtest_results = Analyzer.backtest(swings, method=BT_MODEL, min_history=50)
    
    if not backtest_results.empty:
        scores = Analyzer.verify_performance(backtest_results)
        print(f"\n=== {BT_MODEL} SCORES ===")
        print(f"Direction Accuracy: {scores['direction_accuracy']*100:.1f}%")
        print(f"Price MAPE: {scores['price_mape']*100:.2f}%")
        print(f"Range MAPE: {scores['range_mape']*100:.2f}%")
        print(f"Duration MAPE: {scores['duration_mape']*100:.2f}%")
        display(backtest_results.tail(3))
    else:
        print("No backtest results.")

## 5. Visualization

In [ ]:
if not df.empty:
    fig = go.Figure()
    df_v = df.iloc[max(0, len(df)-600):]
    
    fig.add_trace(go.Candlestick(x=df_v['time'], open=df_v['open'], high=df_v['high'], low=df_v['low'], close=df_v['close'], name='Price', opacity=0.4))
    
    zz_pts = df_v[df_v['zigzag'] != 0]
    fig.add_trace(go.Scatter(x=zz_pts['time'], y=zz_pts['zigzag'], mode='lines+markers', line=dict(color='blue', width=2), name='ZigZag'))

    if prediction:
        fig.add_trace(go.Scatter(x=[swings.iloc[-1]['end_time'], prediction['target_time']], y=[swings.iloc[-1]['end_price'], prediction['target_price']],
                                mode='lines+markers', line=dict(color='magenta', width=3, dash='dot'), name=f'Live {MODEL_TYPE}'))

    if not backtest_results.empty:
        visible_bt = backtest_results[backtest_results['start_time'] >= df_v['time'].iloc[0]]
        bt_x, bt_y = [], []
        for _, r in visible_bt.iterrows():
            bt_x.extend([r['start_time'], r['target_time'], None])
            bt_y.extend([r['start_price'], r['target_price'], None])
        fig.add_trace(go.Scatter(x=bt_x, y=bt_y, mode='lines', line=dict(color='yellow', width=1, dash='dot'), name=f'Hist {BT_MODEL}', opacity=0.4))

    # Signals
    for signals, color, marker, name in [(sot_signals, 'red', 'triangle-down', 'SOT'), (hinge_signals, 'white', 'diamond', 'Hinge'), (springboard_signals, 'gold', 'star', 'Springboard')]:
        if not signals.empty:
            vis = signals[signals['time'] >= df_v['time'].iloc[0]]
            if not vis.empty:
                fig.add_trace(go.Scatter(x=vis['time'], y=vis['price'], mode='markers', marker=dict(color=color, size=14, symbol=marker), name=name))

    fig.update_layout(title=f'{symbol} Advanced Midas (ML & Wave Logic)', template='plotly_dark', xaxis_rangeslider_visible=False, height=800)
    fig.show()